# ✅ Solusi — project-ragkit (Bab 12: RAG)

> Notebook ini memuat versi jadi dari seluruh modul TODO (identik dengan
> `solusi/ragkit_ref.py`), lalu seluruh blok eksperimen dijalankan.
> Bandingkan dengan implementasimu — fokus pada **kenapa**, bukan hanya cocok.


In [ ]:
import sys
from pathlib import Path

# Tambahkan root project ke path (ragkit ada di ./ragkit)
_akar = Path.cwd()
while _akar != _akar.parent and not (_akar / 'ragkit' / 'data.py').is_file():
    _akar = _akar.parent
sys.path.insert(0, str(_akar))

from ragkit.data import DOKUMEN, KUERI_UJI, KUERI_LUAR_CORPUS, TOLOK_UKUR
from ragkit.chunking import buat_semua_chunks, UKURAN_CHUNK, OVERLAP_CHUNK
from ragkit.embed import EncoderEmbedding, cosine
from ragkit.generator import GeneratorEkstraktif, jawab, prompt_rag

print('SEED korpus terkunci. Ukuran chunk:', UKURAN_CHUNK, 'karakter | overlap:', OVERLAP_CHUNK)
print('dokumen:', [d['source'] for d in DOKUMEN])
print('kueri uji:', [q['id'] for q in KUERI_UJI], '+ 1 kueri luar korpus')


### Implementasi lengkap (referensi)

Kontrak sama dengan `tests/`; angka terkunci berasal dari sini.


In [ ]:
import sys
from pathlib import Path

_akar = Path.cwd()
while _akar != _akar.parent and not (_akar / 'ragkit' / 'data.py').is_file():
    _akar = _akar.parent
sys.path.insert(0, str(_akar))
sys.path.insert(0, str(_akar / 'solusi'))

import ragkit_ref as ref
from ragkit import abstain as _am, cache as _cm, evaluate as _em, hybrid as _hm, \
    pipeline as _pm, rerank as _rm, retriever as _rem, vectorstore as _vm

_vm.VectorStore = ref.VectorStore
_rem.buat_retriever = ref.buat_retriever
_rem.evaluasi_retrieval = ref.evaluasi_retrieval
_am.abstaikan = ref.abstaikan
_cm.CacheRetrieval = ref.CacheRetrieval
_cm.retrieve_bercache = ref.retrieve_bercache
_cm.tolok_ukur = ref.tolok_ukur
_hm.cari_bm25 = ref.cari_bm25
_hm.gabung_rrf = ref.gabung_rrf
_hm.cari_hybrid = ref.cari_hybrid
_rm.skorer_kata = ref.skorer_kata
_rm.rerank = ref.rerank
_pm.jalankan_rag = ref.jalankan_rag
_em.evaluasi_ujung_ke_ujung = ref.evaluasi_ujung_ke_ujung

print('implementasi referensi dimuat (solusi/ragkit_ref.py)')


In [ ]:
from ragkit.chunking import buat_semua_chunks
from ragkit.embed import EncoderEmbedding
VectorStore = _vm.VectorStore       # versi referensi (sudah di-patch)
buat_retriever = _rem.buat_retriever
evaluasi_retrieval = _rem.evaluasi_retrieval
abstaikan = _am.abstaikan
CacheRetrieval = _cm.CacheRetrieval
retrieve_bercache = _cm.retrieve_bercache
cari_bm25, gabung_rrf, cari_hybrid = _hm.cari_bm25, _hm.gabung_rrf, _hm.cari_hybrid
skorer_kata, rerank = _rm.skorer_kata, _rm.rerank
jalankan_rag = _pm.jalankan_rag
evaluasi_ujung_ke_ujung = _em.evaluasi_ujung_ke_ujung
chunks = buat_semua_chunks()
retrieve = buat_retriever()
print('korpus:', len(chunks), 'chunk — siap diuji')


## Blok 1 — Vector Store


In [ ]:

chunks = buat_semua_chunks()
enc = EncoderEmbedding()
M = enc.embed([c['text'] for c in chunks])

store = VectorStore()
store.add(chunks, M)

qv = enc.embed_query('Berapa hari cuti tahunan untuk karyawan tetap?')
hasil = store.cari(qv, top_k=3)
for r in hasil:
    print('[%d] %.4f  %s p%d' % (r['chunk']['chunk_index'], r['score'],
                                  r['chunk']['source'], r['chunk']['page']))
    print('     ', r['chunk']['text'][:70].replace('\n', ' '))

# TODO: verifikasi — store kosong tidak crash, tie → index kecil
assert VectorStore().cari([1.0, 2.0], top_k=3) == []
print('OK: store kosong → []')


## Blok 2 — Retriever & hit@k


In [ ]:

retrieve = buat_retriever()

for k in (1, 2, 3):
    ev = evaluasi_retrieval(retrieve, top_k=k)
    print('hit@%d = %.2f | gagal: %s' % (k, ev['hit_rate'], ev['kueri_gagal']))

ev1 = evaluasi_retrieval(retrieve, top_k=1)
assert abs(ev1['hit_rate'] - 0.8) < 1e-9
assert ev1['kueri_gagal'] == ['q1']
print('angka terkunci cocok: hit@1 = 0.8, hit@2 = 1.0')

# Micro-eksperimen: kenapa q1 gagal di top-1?
dapat_q1 = retrieve('Berapa hari cuti tahunan untuk karyawan tetap?', 3)
for r in dapat_q1:
    print('[%d] %.4f %s | %s' % (r['chunk']['chunk_index'], r['score'],
                                  r['chunk']['source'], r['chunk']['text'][:50]))


## Blok 3 — Abstain Gate


In [ ]:
from ragkit.data import SEP_DETEKSI_TIDAK_ADA

# Skor kueri luar korpus (terkunci): 0.0652 — jauh di bawah ambang 0.30
retrieve_q6 = retrieve(KUERI_LUAR_CORPUS['teks'], 1)
skor_q6 = retrieve_q6[0]['score']
print('skor q6 (luar korpus):', round(skor_q6, 4))

h = {'jawaban': '12 hari', 'sitasi': [0], 'sumber': [{'n': 1}],
     'abstain': False, 'skor_terbaik': skor_q6}
out = abstaikan(h, ambang=0.30)
print('hasil gate:', out['jawaban'], '| oleh:', out.get('abstaikan_oleh'))
assert out['abstain'] and out['jawaban'] == SEP_DETEKSI_TIDAK_ADA

# Skor tinggi → utuh, tanpa kunci baru
h2 = dict(h, skor_terbaik=0.59)
out2 = abstaikan(h2, ambang=0.30)
assert out2['jawaban'] == '12 hari' and 'abstaikan_oleh' not in out2
print('OK: gate menimpa bila skor rendah, membiarkan bila aman')

# Micro-eksperimen: coba ambang 0.2 / 0.4 — trade-off apa yang muncul?


## Blok 4 — Cache Konteks


In [ ]:

cache = CacheRetrieval(ttl=10)
panggilan = {'n': 0}

def retrieve_hitung(kueri, top_k=3):
    panggilan['n'] += 1
    return retrieve(kueri, top_k)

pola = []
for i, q in enumerate(TOLOK_UKUR['aturan']):
    r = retrieve_bercache(retrieve_hitung, q, cache, sekarang=i, top_k=3)
    pola.append(r['cache'])
print('pola cache :', pola)
print('pencarian  :', panggilan['n'], 'dari', len(TOLOK_UKUR['aturan']), 'permintaan')
print('statistik  :', cache.statistik())
assert pola == ['miss', 'hit', 'hit', 'miss']
assert panggilan['n'] == 2
print('OK: kueri sama (huruf besar-kecil berbeda) dianggap permintaan sama')

# Micro-eksperimen: kunci cache memakai strip().lower() — kasus apa yang
# masih dianggap 'beda' padahal semantiknya sama? (petunjuk: sinonim)


## Blok 5 — Hybrid Search


In [ ]:

Q = KUERI_UJI[0]['teks']
d_bm25 = cari_bm25(Q, chunks, top_k=3)
print('BM25 top-3 :', [d['chunk']['chunk_index'] for d in d_bm25])

h3 = cari_hybrid(retrieve, chunks, Q, top_k=3)
print('hybrid top-3:', [d['chunk']['chunk_index'] for d in h3])
assert [d['chunk']['chunk_index'] for d in h3] == [5, 0, 4]

hit = sum(1 for q in KUERI_UJI
          if q['source_emas'] in [d['chunk']['source']
                                  for d in cari_hybrid(retrieve, chunks, q['teks'], top_k=3)])
print('hit@3 hybrid:', hit, 'dari', len(KUERI_UJI))
assert hit == 5
print('OK: gabungan vektor + kata kunci paling stabil')


## Blok 6 — Reranker


In [ ]:

Q2 = KUERI_UJI[1]['teks']
dapat = retrieve(Q2, 3)
print('urutan retrieval :', [d['chunk']['chunk_index'] for d in dapat])

dapat_rerank = rerank(Q2, dapat, top_k=2)
print('sesudah rerank   :', [d['chunk']['chunk_index'] for d in dapat_rerank])
assert [d['chunk']['chunk_index'] for d in dapat_rerank] == [3, 4]

# Lihat bobot skor kata: angka × 3
print("skor 'Berapa lama maksimal cuti sakit tanpa surat dokter?' vs")
for c in (chunks[3], chunks[5]):
    print('   [%d] %.1f | %s' % (c['chunk_index'], skorer_kata(Q2, c['text']),
                                  c['text'][:55]))


## Blok 7 — Pipeline


In [ ]:

h = jalankan_rag(retrieve, Q)
print('jawaban   :', h['jawaban'])
print('sumber    :', h['sumber'])
print('skor      :', round(h['skor_terbaik'], 4))
print('biaya     : $%.8f' % h['biaya_usd'])
print('tahap     :', h['tahap'])
assert abs(h['biaya_usd'] - 5.91e-05) < 1e-9
assert h['tahap'] == {'retrieve_ms': 47, 'generate_ms': 50}
print('OK: biaya & latensi per tahap terlapor — bukan "jalan kok"')

# Kueri luar korpus lewat pipeline penuh
h6 = jalankan_rag(retrieve, KUERI_LUAR_CORPUS['teks'])
print('q6        :', h6['jawaban'], '| skor', round(h6['skor_terbaik'], 4))
assert h6['abstain']
print('OK: dua lapis pertahanan — generator + gate')


## Blok 8 — Evaluasi


In [ ]:

ev = evaluasi_ujung_ke_ujung(lambda q: jalankan_rag(retrieve, q))
for p in ev['per_kueri']:
    print('%s benar=%-5s abstain=%-5s %s' % (p['id'], p['benar'], p['abstain'], p['jawaban'][:40]))
print('akurasi       :', ev['akurasi'])
print('abstain_benar :', ev['abstain_benar'], '| jawaban luar:', ev['jawaban_luar'])
assert abs(ev['akurasi'] - 1.0) < 1e-9 and ev['abstain_benar']
print('OK: 5/5 kueri emas + 1 abstain untuk kueri di luar korpus')


## Ringkasan Angka Terkunci

```
chunking      : 3 dokumen → 7 chunk (ukuran 320 / overlap 64 karakter)
hit@1         : 0.8  (q1 gagal: FAQ menang top-1)
hit@2, hit@3  : 1.0
hybrid q1     : top-3 = [5, 0, 4] | hit@3 = 1.0
rerank q2     : [3, 5, 4] → [3, 4]
pipeline q1   : biaya $5.91e-05 | retrieve 47 ms | generate 50 ms
cache 4 req   : miss, hit, hit, miss → 2 pencarian, hit_rate 0.5
e2e           : akurasi 1.0 | abstain benar (skor luar korpus 0.0652)
```
